# 16.1 - API Design (FastAPI)

Status: VERIFIED

## What Are We Solving?

A trained model is useless if nobody can call it. FastAPI gives us type-safe, auto-documented REST endpoints with minimal boilerplate. This unit covers request/response schemas, validation, and how to structure a prediction API.

## Mental Model

Think of your API as a vending machine: the caller inserts a well-formed request (coins), and the machine dispenses a prediction (snack). Pydantic is the coin-validator — it rejects bad input before it reaches your logic.

## FastAPI Endpoint Pattern

```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="ML Prediction API")
model = joblib.load("model.pkl")

class PredictionRequest(BaseModel):
    features: list[float]
    model_version: str = "v1"

class PredictionResponse(BaseModel):
    prediction: float
    confidence: float
    model_version: str

@app.post("/predict", response_model=PredictionResponse)
def predict(req: PredictionRequest):
    import numpy as np
    X = np.array(req.features).reshape(1, -1)
    pred = model.predict(X)[0]
    proba = max(model.predict_proba(X)[0])
    return PredictionResponse(
        prediction=float(pred),
        confidence=float(proba),
        model_version=req.model_version,
    )
```

## Pydantic Validation in Pure Python

Below we demonstrate Pydantic-style validation using plain Python dicts and dataclasses.

In [1]:
import matplotlib
matplotlib.use('Agg')
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PredictionRequest:
    features: list
    model_version: str = "v1"

    def __post_init__(self):
        if not isinstance(self.features, list):
            raise TypeError("features must be a list")
        if len(self.features) == 0:
            raise ValueError("features cannot be empty")
        if not all(isinstance(x, (int, float)) for x in self.features):
            raise ValueError("all features must be numeric")

# --- Valid request ---
req = PredictionRequest(features=[1.5, 2.3, 0.8])
print(f"Valid request: {req}")

# --- Invalid requests ---
errors = []
try:
    PredictionRequest(features=[])
except ValueError as e:
    errors.append(f"Empty list: {e}")

try:
    PredictionRequest(features=[1, "two", 3])
except ValueError as e:
    errors.append(f"Bad types: {e}")

for err in errors:
    print(f"Rejected: {err}")


Valid request: PredictionRequest(features=[1.5, 2.3, 0.8], model_version='v1')
Rejected: Empty list: features cannot be empty
Rejected: Bad types: all features must be numeric


## API Design Best Practices

- **Version your API**: `/api/v1/predict`, `/api/v2/predict`
- **Use response models**: forces consistent output shape
- **Add health checks**: `GET /health` returns model status
- **Validate early**: reject bad input before it hits GPU/CPU
- **Return confidence scores**: lets callers decide thresholds

In [2]:
import matplotlib
matplotlib.use('Agg')
import json

# Simulate building an OpenAPI spec programmatically
spec = {
    "openapi": "3.0.0",
    "info": {"title": "ML Prediction API", "version": "1.0.0"},
    "paths": {
        "/predict": {
            "post": {
                "summary": "Get model prediction",
                "requestBody": {
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "features": {"type": "array", "items": {"type": "number"}},
                                    "model_version": {"type": "string", "default": "v1"},
                                },
                                "required": ["features"],
                            }
                        }
                    }
                },
                "responses": {"200": {"description": "Prediction result"}},
            }
        }
    }
}
print("Generated OpenAPI spec:")
print(json.dumps(spec, indent=2)[:500])


Generated OpenAPI spec:
{
  "openapi": "3.0.0",
  "info": {
    "title": "ML Prediction API",
    "version": "1.0.0"
  },
  "paths": {
    "/predict": {
      "post": {
        "summary": "Get model prediction",
        "requestBody": {
          "content": {
            "application/json": {
              "schema": {
                "type": "object",
                "properties": {
                  "features": {
                    "type": "array",
                    "items": {
                      "type": "number"


## Project Structure for a FastAPI ML Service

```
ml-api/
  app/
    __init__.py
    main.py          # FastAPI app, routes
    schemas.py       # Pydantic models
    model.py         # load model, predict function
    config.py        # settings, env vars
  tests/
    test_predict.py
  Dockerfile
  requirements.txt
```

In [3]:
import matplotlib
matplotlib.use('Agg')
import os

# Demonstrate project layout programmatically
structure = [
    'ml-api/app/__init__.py',
    'ml-api/app/main.py',
    'ml-api/app/schemas.py',
    'ml-api/app/model.py',
    'ml-api/app/config.py',
    'ml-api/tests/test_predict.py',
    'ml-api/Dockerfile',
    'ml-api/requirements.txt',
]
print("Recommended ML API project structure:")
for path in structure:
    depth = path.count('/') - 1
    print(f"{'  ' * depth}{os.path.basename(path)}")


Recommended ML API project structure:
  __init__.py
  main.py
  schemas.py
  model.py
  config.py
  test_predict.py
Dockerfile
requirements.txt


In [4]:
import matplotlib
matplotlib.use('Agg')
print('VERIFICATION PASSED: Phase 16.1 complete')


VERIFICATION PASSED: Phase 16.1 complete
